# Analyse BN des scénarios récurrents — metallurgie

Cette analyse commence après la sélection des partitions Pareto dans le
notebook `topic_modeling_results_metallurgie.ipynb`. Elle ne relance ni UMAP,
ni HDBSCAN, ni le resampling. Une configuration Pareto est choisie explicitement
pour chaque rôle, puis un seul réseau bayésien latent est ajusté par Structural EM.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SCENARIO_DIR = Path.cwd() / "text" / "recurrent_scenarios"
if not (SCENARIO_DIR / "scenario_pipeline.py").is_file():
    SCENARIO_DIR = Path.cwd()
if str(SCENARIO_DIR) not in sys.path:
    sys.path.insert(0, str(SCENARIO_DIR))

from scenario_pipeline import (
    load_yaml_config,
    select_dataset_config,
    resolve_config_paths,
    load_units,
    run_frozen_bn_analysis,
)


## 1. Configuration des partitions BN

Modifie uniquement les identifiants ci-dessous pour analyser d'autres
partitions Pareto. Il faut exactement une configuration par rôle. Ces choix
n'entraînent aucune nouvelle étape de clustering.


In [ ]:
DATASET_ID = 'metallurgie'
DISCOVERY_RUN_NAME = "theme_discovery_audit"
RUN_DIR = SCENARIO_DIR / "runs" / DISCOVERY_RUN_NAME / DATASET_ID
BN_OUTPUT_DIR = RUN_DIR / "bayesian_networks"

BN_PARTITION_SELECTIONS = {'A0': 'A0_cfg_007', 'A1': 'A1_cfg_010', 'B': 'B_cfg_011', 'C': 'C_cfg_019'}

CONFIG_PATH = SCENARIO_DIR / "config.yaml"
config = resolve_config_paths(
    select_dataset_config(load_yaml_config(CONFIG_PATH), DATASET_ID),
    CONFIG_PATH,
)
config["bayesian_networks"]["min_theme_support_count"] = 20
config["bayesian_networks"]["d_max"] = 2
config["bayesian_networks"]["latent_states"] = list(range(2, 9))
config["bayesian_networks"]["n_initializations"] = 20
config["bayesian_networks"]["alpha"] = 0.5

print("Dataset:", DATASET_ID)
print("Partitions:", BN_PARTITION_SELECTIONS)


## 2. Matrice accident × facteurs figés


In [ ]:
units, _ = load_units(config)
analysis = run_frozen_bn_analysis(
    config=config,
    run_dir=RUN_DIR,
    partition_selections=BN_PARTITION_SELECTIONS,
    output_dir=BN_OUTPUT_DIR,
    units=units,
)

matrix = analysis["matrix"]
theme_dictionary = analysis["theme_dictionary"]
excluded_themes = analysis["excluded_themes"]
print("Accidents:", len(matrix), "Variables BN:", len(theme_dictionary))
display(theme_dictionary)
display(excluded_themes)


## 3. Sélection de K par BIC


In [ ]:
k_selection = analysis["selection"]
display(k_selection.sort_values(["K", "bic"]))
selected_result = analysis["result"]
print("K sélectionné:", selected_result.n_states)


In [ ]:
fig, axis = plt.subplots(figsize=(8, 5))
summary = k_selection.groupby("K", as_index=False)["bic"].min()
axis.plot(summary["K"], summary["bic"], marker="o")
axis.set(xlabel="Nombre de familles latentes K", ylabel="BIC", title="Sélection de K par BIC")
axis.grid(alpha=0.25)
fig.tight_layout()
display(fig)


## 4. Familles latentes, profils et scénarios


In [ ]:
display(analysis["profiles"].head(50))
display(analysis["scenarios"])
display(analysis["supports"])
display(analysis["prototypes"])


In [ ]:
profiles = analysis["profiles"].pivot(index="variable_name", columns="family_id", values="probability")
fig, axis = plt.subplots(figsize=(10, max(5, len(profiles) * 0.22)))
image = axis.imshow(profiles.fillna(0).to_numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=1)
axis.set(yticks=np.arange(len(profiles)), yticklabels=profiles.index, xlabel="Famille latente", title="Profils factoriels P(X=1 | Z)")
fig.colorbar(image, ax=axis, label="Probabilité")
fig.tight_layout()
display(fig)


## Fichiers produits

Les résultats sont écrits dans `RUN_DIR / "bayesian_networks"`, notamment la
matrice multi-hot, les responsabilités postérieures, les profils familiaux,
les CPT finales, les arêtes apprises, les supports et les prototypes observés.
